# BorakBot — Speech Evaluation (Colab)

Run the cells **in order, top to bottom**. Whenever the Colab runtime disconnects,
start again from Cell 1 — nothing in `/content` survives a restart.

    code    -> GitHub
    audio   -> Google Drive
    results -> Google Drive

Anything living only in `/content` is temporary.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**Save this notebook**: File → Save a copy in Drive, so you don't rebuild it each time.

## Cell 1 — Setup

Installs packages, mounts Drive, clones the repo, copies the 48 audio files in.
Takes 2–3 minutes. Run this after every runtime restart.

In [ ]:
!pip install -q openai-whisper transformers jiwer

from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/NLP-BorakBot
!git clone -q https://github.com/yongvay/NLP-BorakBot.git /content/NLP-BorakBot

import shutil, pathlib
SRC = pathlib.Path("/content/drive/MyDrive/RDS3S1/NLP/NLP Assignment/Audio")
DST = pathlib.Path("/content/NLP-BorakBot/stage1_stt/audio")
DST.mkdir(parents=True, exist_ok=True)

if not SRC.exists():
    print("!! audio folder not found:", SRC)
    print("   Run Cell 1b to locate it.")
else:
    for w in SRC.rglob("*.wav"):          # rglob descends into speaker subfolders
        shutil.copy2(w, DST / w.name)
    n = len(list(DST.glob("*.wav")))
    print(f"{n} audio files ready", "OK" if n == 48 else "<-- EXPECTED 48")

## Cell 1b — Only if Cell 1 said the audio folder was not found

Searches your whole Drive and prints where the `.wav` files actually are. Copy that
path into `SRC` in Cell 1 and re-run Cell 1.

Note: files in *Shared with me* are not mounted. If Pei Qi shared the folder, open
Drive in a browser, right-click it → Organise → Add shortcut to Drive.

In [ ]:
import pathlib
from collections import Counter

hits = Counter()
for p in pathlib.Path("/content/drive").rglob("*"):
    if p.suffix.lower() in {".wav", ".m4a", ".mp4"}:
        hits[(str(p.parent), p.suffix.lower())] += 1

for (folder, ext), n in hits.most_common(20):
    print(f"{n:>4} {ext}   {folder}")

## Cell 2 — Run the full evaluation

All nine configurations across 48 clips. Roughly 15 minutes on a T4.

Downloads two models on first run (~950 MB total), cached afterwards.

In [ ]:
%cd /content/NLP-BorakBot
!python stage1_stt/run_wer.py

## Cell 3 — Save results to Drive

**Do not skip this.** Results are written inside `/content` and vanish when the
runtime disconnects. This is what makes them survive.

In [ ]:
import shutil, pathlib
OUT = pathlib.Path("/content/drive/MyDrive/RDS3S1/NLP/NLP Assignment/results")
OUT.mkdir(parents=True, exist_ok=True)
SRC = pathlib.Path("/content/NLP-BorakBot/stage1_stt")

for f in ["results_detail.csv", "results_summary.csv",
          "results_by_category.csv", "substitutions.csv",
          "detector_probs.csv", "threshold_sweep.csv"]:
    if (SRC / f).exists():
        shutil.copy2(SRC / f, OUT / f)
    else:
        print("  (skipped, not generated this run):", f)

print("saved to Drive:")
for p in sorted(OUT.iterdir()):
    print("  ", p.name)

## Cell 4 — Inspect transcripts (optional)

Reads the cached results, so it's instant and needs no GPU. A WER table tells you
something is wrong; only the transcripts tell you what.

In [ ]:
%cd /content/NLP-BorakBot

# worst clips in the best configuration
!python stage1_stt/inspect_results.py

# a single category
!python stage1_stt/inspect_results.py --category en_dom

# which language was chosen per category
!python stage1_stt/inspect_results.py --config malaysian_prompt_routed --langs

## Cell 5 — Rescore without re-transcribing (optional)

After editing `stage1_stt/orthography_map.json`, this recomputes lenient WER from the
cached transcripts in seconds. Push your edit to GitHub first, then:

In [ ]:
%cd /content/NLP-BorakBot
!git pull
!python stage1_stt/run_wer.py --score-only

## Cell 6 — Language-threshold sensitivity (optional, no GPU)

`run_wer.py` logs the router's *decision* (`en`/`ms`). It now also logs the
*evidence* — `p_en`, the detector's English probability. The decision alone cannot
tell a clip missed narrowly (cutoff too high) from one missed completely (detector
deaf to the English), and those imply opposite conclusions in the report.

The first command runs the detector only: one encoder pass per clip, no
transcription. The second reconstructs routed WER at every threshold by *selecting*
between the already-cached forced-Malay and forced-English transcripts, so nothing
is re-decoded.

**Read the output as a sensitivity curve, not a tuning run.** Adopting the
minimising threshold would fit the decision rule to the same 48 clips it is scored
on. Report the curve; keep the a priori threshold.


In [ ]:
%cd /content/NLP-BorakBot

!python stage1_stt/run_wer.py --detect-only
!python stage1_stt/threshold_sweep.py


---

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `No such file or directory: /content/NLP-BorakBot` | runtime restarted | re-run Cell 1 |
| `0 files ready` | wrong Drive path, or folder is *Shared with me* | run Cell 1b |
| `expected 48, missing N` | audio not fully uploaded | check Drive folder |
| results CSV disagrees with the printed table | stale copy in Drive | re-run Cell 3 |
| code changes not taking effect | Colab has the old clone | re-run Cell 1, or `!git pull` |

**After a code change on your laptop**, always: `git add -A`, `git commit`, `git push`
— then `!git pull` in Colab. Colab reads code from GitHub, never from your computer.